# NLP-32 — Khảo sát dữ liệu UIT-ViCTSD (EDA)

Chạy notebook từ thư mục project, sau khi tải dữ liệu có quyền sử dụng. Không sao chép corpus vào báo cáo hoặc kho mã công khai.

**Không dùng các số liệu EDA này để chọn test threshold.**


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
PROJECT = Path.cwd().resolve()
if not (PROJECT / 'src').exists():
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))
from src.data_loader import load_split, leakage_report
from src.config import RAW
print(leakage_report(RAW))


## 1. Kiểm tra schema, nhãn và văn bản


In [ ]:
frames = {s: load_split(RAW,s) for s in ['train','valid','test']}
for name,df in frames.items():
    print(name, 'n=',len(df), 'ratio=',df['label'].value_counts(normalize=True).sort_index().to_dict())
    display(df.assign(char_len=df['text'].str.len())[['id','label','char_len']].head())


## 2. Phân bố nhãn, độ dài bình luận


In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4))
for s,df in frames.items():
    counts=df['label'].value_counts().reindex([0,1],fill_value=0)
    axs[0].plot([0,1],counts,marker='o',label=s)
    axs[1].hist(df['text'].str.len(), bins=30, alpha=.4,label=s)
axs[0].set_xticks([0,1],['Không độc hại','Độc hại'])
axs[0].set_ylabel('Số lượng')
axs[0].set_title('Phân bố nhãn theo split')
axs[1].set_title('Độ dài ký tự bình luận')
axs[1].set_xlabel('Số ký tự')
for a in axs:a.legend()
plt.tight_layout();plt.show()


## 3. Trùng lặp, kiểm tra rò rỉ dữ liệu


In [ ]:
for s,df in frames.items():
    dupe = int(df['text'].str.casefold().duplicated().sum())
    print(s,'Số văn bản trùng trong split:',dupe)
print('Giao nhau giữa các split:', leakage_report(RAW)['same_text_cross_split'])
print('Nếu có trùng, nêu trong báo cáo trước khi chọn protocol xử lý.')


## 4. Nhật ký nguồn và kiểm tra thủ công

Ghi URL, ngày tải, điều khoản, phiên bản/split và số mẫu. Với mẫu cần xem xét, chỉ truy cập dữ liệu trong môi trường được cấp phép; không xuất bình luận nguyên văn trong báo cáo.
